# Численное исследование системы Лоренца

Последовательно выполните все ячейки. Используются те же функции, что и в main.py. Полная теория и интерпретация чисел — в [report.md](report.md). Первые два метода самостоятельные, RK45 библиотечный. При запуске в Jupyter выберите ядро с установленными зависимостями.

## Математические понятия
[Отдельный справочник](math-glossary.md): производные, ОДУ, задача Коши, Эйлер, RK4/RK45, ошибки, устойчивость, Якобиан, хаос и анализ данных.

In [1]:
from pathlib import Path
import json
from experiments import load_config, run_experiments
from visualization import render_figures, render_animation
from reporting import build_report
config = load_config()
output = Path('results')
print(json.dumps(config, ensure_ascii=False, indent=2))

{
  "sigma": 10.0,
  "rho": 28.0,
  "beta": 2.6666666666666665,
  "initial_state": [
    1.0,
    1.0,
    1.0
  ],
  "perturbation": [
    1e-08,
    0.0,
    0.0
  ],
  "t_start": 0.0,
  "t_end": 40.0,
  "step": 0.01,
  "short_end": 2.0,
  "steps": [
    0.04,
    0.02,
    0.01,
    0.005,
    0.0025
  ],
  "rk45_rtol": 1e-07,
  "rk45_atol": 1e-09,
  "reference_rtol": 1e-12,
  "reference_atol": 1e-14,
  "slope_window": [
    10.0,
    25.0
  ],
  "transient_end": 5.0,
  "timing_repeats": 3,
  "animation_frames": 160,
  "animation_fps": 20,
  "dpi": 170,
  "euler_refinement_steps": [
    0.00125,
    0.000625,
    0.0003125,
    0.00015625
  ]
}


## Модель и равновесия

Система: x′=σ(y−x), y′=x(ρ−z)−y, z′=xy−βz. Все компоненты имеют начальные значения. Матрица Якоби описывает изменение скоростей при малом изменении состояния.

In [2]:
from lorenz import Parameters, rhs, jacobian, equilibria
import numpy as np
p = Parameters(config['sigma'], config['rho'], config['beta'])
print('f(u0) =', rhs(0, config['initial_state'], p))
for name, u in equilibria(p).items():
    print(name, u, 'спектр:', np.linalg.eigvals(jacobian(u,p)))

f(u0) = [ 0.         26.         -1.66666667]
O [0. 0. 0.] спектр: [-22.82772345  11.82772345  -2.66666667]
C+ [ 8.48528137  8.48528137 27.        ] спектр: [-13.85457791 +0.j           0.09395562+10.19450522j
   0.09395562-10.19450522j]
C- [-8.48528137 -8.48528137 27.        ] спектр: [-13.85457791 +0.j           0.09395562+10.19450522j
   0.09395562-10.19450522j]


## Численные эксперименты

Эталон RK45 проверяется DOP853 и более строгим RK45 на коротком интервале. Основные методы сравниваются на общей сетке. Переполнение крупного шага Эйлера фиксируется отдельно.

In [3]:
data = run_experiments(config, output)
for row in data['summary']['metrics']:
    print(row)
print('Контроль эталона:', data['summary']['reference'])

{'method': 'euler', 'max_error': 46.30011925725157, 'final_error': 4.39938848948866, 'short_max_error': 20.340333461930033, 'seconds_median': 0.0139381129999947, 'n_steps': 4000, 'nfev': 4000, 'n_output': 4001}
{'method': 'rk4', 'max_error': 50.280371710125074, 'final_error': 21.78461847989416, 'short_max_error': 0.0007982399886875303, 'seconds_median': 0.049342638000098304, 'n_steps': 4000, 'nfev': 16000, 'n_output': 4001}
{'method': 'RK45', 'max_error': 52.313970257301605, 'final_error': 15.826516959307925, 'short_max_error': 5.785202944671716e-06, 'seconds_median': 0.08946843399985482, 'n_steps': 2667, 'nfev': 17036, 'n_output': 4001}
Контроль эталона: {'rtol': 1e-12, 'atol': 1e-14, 'n_steps': 25975, 'nfev': 156128, 'short_dop853_disagreement': 9.584398515815454e-11, 'short_tighter_rk45_disagreement': 9.288314676896231e-11}


## Сходимость и чувствительность

Первый порядок Эйлера проверяется также на дополнительной мелкой серии. Наклон ln расстояния зависит от окна и не объявляется точным показателем Ляпунова.

In [4]:
for row in data['convergence']:
    print(row)
print('Порядки исходной серии:', data['summary']['orders_last_three'])
print('Уточнённый порядок Эйлера:', data['summary']['euler_refinement_order'])
print('Чувствительность:', data['summary']['sensitivity'])

{'method': 'euler', 'h': 0.04, 'max_error': '', 'final_error': '', 'observed_order': '', 'status': 'nonfinite'}
{'method': 'euler', 'h': 0.02, 'max_error': 36.62620310541604, 'final_error': 6.338107986801016, 'observed_order': '', 'status': 'ok'}
{'method': 'euler', 'h': 0.01, 'max_error': 20.34033346192887, 'final_error': 20.34033346192887, 'observed_order': 0.8485328191617233, 'status': 'ok'}
{'method': 'euler', 'h': 0.005, 'max_error': 5.688695271584314, 'final_error': 4.859289902031129, 'observed_order': 1.8381736242659037, 'status': 'ok'}
{'method': 'euler', 'h': 0.0025, 'max_error': 2.2754166203769803, 'final_error': 2.0407145223738787, 'observed_order': 1.3219670801016408, 'status': 'ok'}
{'method': 'rk4', 'h': 0.04, 'max_error': 0.19563127184521859, 'final_error': 0.1439544390524911, 'observed_order': '', 'status': 'ok'}
{'method': 'rk4', 'h': 0.02, 'max_error': 0.012901279576491865, 'final_error': 0.004761027186012198, 'observed_order': 3.9225509374649294, 'status': 'ok'}
{'me

In [5]:
render_figures(data, config, output)
print('Сохранены семь PNG.')

Сохранены семь PNG.


![Аттрактор](results/lorenz_attractor.png)

![Сравнение](results/methods_comparison.png)

![Зависимость от шага](results/step_dependence.png)

![Чувствительность](results/sensitivity.png)

![Равновесия](results/equilibria.png)

## Анимация и отчёт

Следующая ячейка пересоздаёт GIF; GUI и ffmpeg не нужны. После изменения параметров важно выполнить её заново.

In [6]:
render_animation(data, config, output)
build_report(data['summary'], output)
print('GIF и отчёт сохранены.')

GIF и отчёт сохранены.


![Две траектории](results/animation.gif)

[Полный отчёт](report.md) · [Подготовка к защите](plan-zashchity.md)

In [7]:
import subprocess, sys
completed = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], capture_output=True, text=True)
print(completed.stdout + completed.stderr)
completed.check_returncode()

test_equilibria_and_spectra (test_numerics.NumericsTests.test_equilibria_and_spectra) ... ok
test_euler_known_step (test_numerics.NumericsTests.test_euler_known_step) ... ok
test_euler_order (test_numerics.NumericsTests.test_euler_order) ... ok
test_grid_and_evaluation_counts (test_numerics.NumericsTests.test_grid_and_evaluation_counts) ... ok
test_jacobian_independent_difference (test_numerics.NumericsTests.test_jacobian_independent_difference) ... ok
test_overflow_not_silent (test_numerics.NumericsTests.test_overflow_not_silent) ... ok
test_rhs_known_value (test_numerics.NumericsTests.test_rhs_known_value) ... ok
test_rk4_exponential (test_numerics.NumericsTests.test_rk4_exponential) ... ok
test_rk4_nonautonomous (test_numerics.NumericsTests.test_rk4_nonautonomous) ... ok
test_rk4_order (test_numerics.NumericsTests.test_rk4_order) ... ok
test_short_agreement (test_numerics.NumericsTests.test_short_agreement) ... ok
test_standard_trajectories_finite (test_numerics.NumericsTests.test_s

## Вывод

Кратковременная сходимость и долгосрочное поведение — разные критерии. Неустойчивость равновесий подтверждается спектрами, а численный хаос исследуется с оговорками. Численные выводы следует сверять с последним summary.json; пример одногруппника ещё не изучен.

## Защита численных констант
[Паспорт чисел](constants-explained.md) · [Реальные сравнительные таблицы](constants-results.md) · [Построчный индекс](constants-index.md).
Дополнительный запуск из терминала: `python parameter_study.py`, затем `python build_constant_index.py`. Исходные параметры не выдаются за экспериментально оптимальные.